# Exp1 · query token reference từ PlanarPromptFlow của 3Qformer+pfm

Bật GPU và Internet trên Kaggle, gắn FashionIQ đã giải nén và chọn **Run All**. Có thể gắn thêm `blip2_pretrained_vitL.pth`. Notebook hiển thị commit trainer và kiểm tra mã nguồn trước train. Mặc định 30 epoch, batch 16, seed 1000, gallery sau epoch cuối.

Module từ `3Qformer+pfm@237c98d` nhận đặc trưng ảnh và caption trước Q-Former. Decoder xuất 32 × 768 giá trị. Bias của decoder được khởi tạo một lần từ bộ query pretrained, cho đầu ra ban đầu bằng Exp1. Khi train, dòng reference nhận query trực tiếp từ PFM. Caption và ảnh reference đi chung Q-Former. Target dùng query pretrained, train dùng contrastive loss của Exp1.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import shutil
import hashlib

RUNS = ['reference_3qpfm']
DATA_ROOT = None  # Path('/kaggle/input/.../fashionIQ') nếu tự tìm không được
CHECKPOINT = None  # Checkpoint BLIP-2 pretrained, không phải checkpoint đã train
RESUME_FROM = {}  # Ví dụ {RUNS[0]: '/kaggle/input/run/last.pt'}
SEED = 1000
BATCH_SIZE = 16
EPOCHS = 30  # Tổng số epoch của toàn bộ thí nghiệm
EPOCHS_PER_SESSION = None  # Ví dụ: 3 để dừng sau 3 epoch mới, lưu output rồi resume
assert EPOCHS > 0 and (EPOCHS_PER_SESSION is None or EPOCHS_PER_SESSION > 0)
RUN_TAG = 's1'  # Đổi tag để bắt đầu lượt mới với cấu hình khác
WORK = Path('/kaggle/working')
EXPERIMENT_SHA = 'b36607df9d7813167df9cf2ef1d1de605938f1f2'
BACKBONE_SHA = 'c1c7f6b62fe6d556b9554b6c7cae111f71f67917'
RUNNER_SHA = 'a518480af7cb6c2c3d999d02eb88861d5991c246'
RUNNER = WORK / 'exp1_3qpfm_runner_source'
PACKAGE = RUNNER / 'experiments' / 'exp1_3qpfm_reference'
BACKBONE = WORK / 'qure_main_backbone'
CODE = WORK / 'exp1_3qpfm_reference_code'
CODE.mkdir(exist_ok=True)
assert BATCH_SIZE >= 2


## Cài thư viện và lấy source đã khóa commit

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU in Kaggle settings.")
subprocess.run([sys.executable, "-m", "pip", "install", "--prefer-binary", "--only-binary=tokenizers",
                "transformers==4.46.3", "tokenizers==0.20.3",
                "timm==0.9.16", "fairscale", "omegaconf", "iopath"], check=True)

# Verify dependencies in a fresh process, as used by the training runner.
subprocess.run([sys.executable, "-c", """
import sys, torch, transformers, tokenizers
from transformers.modeling_utils import (
    PreTrainedModel, apply_chunking_to_forward,
    find_pruneable_heads_and_indices, prune_linear_layer,
)
assert transformers.__version__ == "4.46.3"
assert tokenizers.__version__ == "0.20.3"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__, "| tokenizers:", tokenizers.__version__)
"""], check=True)

def checkout(repo, sha, destination):
    if not (destination / ".git").is_dir():
        destination.mkdir(exist_ok=True)
        subprocess.run(["git", "init", "-q", str(destination)], check=True)
    probe = subprocess.run(["git", "-C", str(destination), "rev-parse", "--verify", "HEAD"], capture_output=True, text=True)
    if probe.stdout.strip() != sha:
        subprocess.run(["git", "-C", str(destination), "fetch", "--depth=1", repo, sha], check=True)
        subprocess.run(["git", "-C", str(destination), "checkout", "--detach", sha], check=True)
    print(destination.name, subprocess.check_output(["git", "-C", str(destination), "rev-parse", "HEAD"], text=True).strip())

checkout("https://github.com/fast2104/QuRe.git", RUNNER_SHA, RUNNER)
checkout("https://github.com/fast2104/QuRe.git", BACKBONE_SHA, BACKBONE)

## Dataset và checkpoint pretrained

In [ ]:
import shutil
inputs = Path("/kaggle/input")
if DATA_ROOT is None:
    matches = [p.parent.parent for p in inputs.rglob("cap.dress.train.json")
               if (p.parent.parent / "images").is_dir() and (p.parent.parent / "image_splits").is_dir()]
    if len(matches) != 1:
        raise ValueError(f"Set DATA_ROOT to the extracted FashionIQ directory. Candidates: {matches}")
    DATA_ROOT = matches[0]
DATA_ROOT = Path(DATA_ROOT)
for category in ("dress", "shirt", "toptee"):
    for split in ("train", "val"):
        annotations = DATA_ROOT / "captions" / f"cap.{category}.{split}.json"
        sample = json.loads(annotations.read_text())[0]
        for key in ("candidate", "target"):
            image = DATA_ROOT / "images" / (sample[key] + ".png")
            if not image.is_file():
                raise FileNotFoundError(image)
    assert (DATA_ROOT / "image_splits" / f"split.{category}.val.json").is_file()

name = "blip2_pretrained_vitL.pth"
if CHECKPOINT is None:
    CHECKPOINT = next(inputs.rglob(name), WORK / name)
CHECKPOINT = Path(CHECKPOINT).resolve()
if not CHECKPOINT.is_file():
    if CHECKPOINT != WORK / name:
        raise FileNotFoundError(CHECKPOINT)
    base_url = "https://storage.googleapis.com/sfr-vision-language-research/LAVIS/models/BLIP2/"
    temporary = WORK / (name + ".part")
    subprocess.run(["curl", "-fL", "--retry", "3", "-o", str(temporary), base_url + name], check=True)
    temporary.replace(CHECKPOINT)
if CHECKPOINT.name != name:
    raise ValueError(f"Expected checkpoint filename {name}")
os.environ["QURe_CHECKPOINT_ROOT"] = str(CHECKPOINT.parent)
print("FashionIQ:", DATA_ROOT, "| BLIP-2:", CHECKPOINT)

## Chuẩn bị backbone

In [ ]:
def original(rel):
    return subprocess.check_output(["git", "-C", str(BACKBONE), "show", f"HEAD:{rel}"], text=True)

model_init = "lavis/models/__init__.py"
source = original(model_init)
loaders = source[source.index("def load_model("):source.index("class ModelZoo")]
imports = """import logging
import torch
from omegaconf import OmegaConf
from lavis.common.registry import registry
from lavis.models.base_model import BaseModel
from lavis.processors import BaseProcessor
from lavis.models.blip2_models.blip2_qure import Blip2QuRe
"""
(BACKBONE / model_init).write_text(imports + "\n" + loaders)
(BACKBONE / "lavis/processors/__init__.py").write_text(
    "from lavis.processors.base_processor import BaseProcessor\nfrom lavis.processors import blip_processors\n")
(BACKBONE / "data/__init__.py").write_text("# Retrieval runner imports only data.utils.\n")
yaml_path = "lavis/configs/models/blip2/blip2_pretrain_vitL.yaml"
import re
yaml_text, count = re.subn(r"(?m)^  pretrained:.*$", lambda _: "  pretrained: " + json.dumps(str(CHECKPOINT)), original(yaml_path))
assert count == 1
(BACKBONE / yaml_path).write_text(yaml_text)

## Mã nguồn Exp1 và PFM đã ghim commit

In [ ]:
source_files = ['prompt_flow.py', 'reference_pfm.py', 'model.py', 'data.py', 'train_pfm.py', 'preflight.py']
for filename in source_files:
    shutil.copyfile(PACKAGE / filename, CODE / filename)
flow_object = subprocess.check_output(
    ["git", "hash-object", "--no-filters", str(CODE / "prompt_flow.py")], text=True).strip()
assert flow_object == "fc38b1a3622e2e71e96c179687fcd30d65f37fb0", flow_object
print("PlanarPromptFlow verified:", flow_object)

## Train, lưu checkpoint và đánh giá ở epoch cuối
Mỗi epoch hoàn tất sẽ ghi `last.pt`. Gallery và Recall chỉ chạy sau khi hoàn tất tổng số `EPOCHS`, kể cả khi chia nhiều phiên. Với checkpoint cuối đã train xong nhưng chưa có Recall, resume sẽ chạy phần validation.

Để chia lượt 30 epoch, giữ `EPOCHS = 30` và chọn `EPOCHS_PER_SESSION` theo thời gian GPU của bạn. Sau khi phiên kết thúc, lưu output. Ở phiên mới, gắn output cũ làm Kaggle Input và đặt `RESUME_FROM = {RUNS[0]: '/kaggle/input/.../last.pt'}` trong cell cấu hình. Nếu đường dẫn output hiện tại đã có `last.pt`, notebook ưu tiên checkpoint đó.

In [ ]:
template = json.loads((PACKAGE / 'config.json').read_text())
config_paths = {}
output_paths = {}
source_digest = hashlib.sha256(''.join((CODE / f).read_text(encoding='utf-8') for f in source_files).encode()).hexdigest()
assert source_digest == template['source_sha256']
for variant in RUNS:
    output = WORK / f'exp1_suite_{variant}_seed{SEED}_{RUN_TAG}'
    pfm = template['pfm']
    assert variant == pfm['variant']
    config = dict(template, seed=SEED, batch_size=BATCH_SIZE, epochs=EPOCHS, evaluate_every=EPOCHS,
                  output_dir=str(output), pfm=pfm, source_sha256=source_digest,
                  experiment_sha=EXPERIMENT_SHA, backbone_sha=BACKBONE_SHA)
    config_path = CODE / f'config_{variant}.json'
    config_path.write_text(json.dumps(config, indent=2))
    config_paths[variant] = config_path
    output_paths[variant] = output
    resume = output / 'last.pt'
    if not resume.is_file() and variant in RESUME_FROM:
        resume = Path(RESUME_FROM[variant])
        if not resume.is_file():
            raise FileNotFoundError(resume)
    if resume.is_file():
        print(f'[RESUME] {resume}', flush=True)
        output.mkdir(exist_ok=True)
        if resume.parent.resolve() != output.resolve():
            for filename in ('history.json', 'run_summary.json', 'latest_metrics.json', 'best.pt'):
                source = resume.parent / filename
                if source.exists():
                    shutil.copy2(source, output / filename)
    elif (output / 'history.json').exists():
        raise RuntimeError(f'History exists without last.pt in {output}; select a new RUN_TAG.')
    subprocess.run([sys.executable, str(CODE / 'preflight.py'), str(config_path),
                    str(BACKBONE), str(DATA_ROOT)], cwd=CODE, check=True)
    output.mkdir(exist_ok=True)
    for filename in source_files:
        shutil.copy2(CODE / filename, output / filename)
    command = [sys.executable, '-u', str(CODE / 'train_pfm.py'),
               '--config', str(config_path), '--qure-root', str(BACKBONE),
               '--fashioniq-root', str(DATA_ROOT), '--output-dir', str(output)]
    if EPOCHS_PER_SESSION is not None:
        command += ['--epochs-per-session', str(EPOCHS_PER_SESSION)]
    if resume.is_file():
        command += ['--resume', str(resume)]
    with (output / 'console.log').open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


## Recall trên FashionIQ

In [ ]:
import pandas as pd
rows = []
for variant, output in output_paths.items():
    history = json.loads((output / 'history.json').read_text())
    evaluated = [r for r in history if 'average_recall' in r]
    if history and history[-1]['epoch'] < EPOCHS:
        print(f"Đã lưu epoch {history[-1]['epoch']}/{EPOCHS}. Gallery sẽ chạy sau epoch {EPOCHS}.")
    if evaluated:
        last = evaluated[-1]
        best = max(evaluated, key=lambda r: r['average_recall'])
        rows.append(dict(variant=variant, last_epoch=last['epoch'],
                         R10=last['average_recall_at10'], R50=last['average_recall_at50'],
                         Avg=last['average_recall'], best_epoch=best['epoch'], best_Avg=best['average_recall']))
summary = pd.DataFrame(rows)
display(summary.round(3))
summary.to_csv(CODE / 'results_summary.csv', index=False)
